# HR Analytics Project — Data Cleaning

## Objective
This notebook cleans the raw HR dataset after the profiling step.

The cleaning process includes:
- Removing duplicate records
- Standardizing department names
- Standardizing job titles
- Cleaning employee status values
- Converting date columns
- Converting numeric columns
- Handling invalid salaries
- Normalizing pay_frequency labels and deriving annual_salary_usd
- Handling invalid performance and satisfaction scores
- Cleaning training and attendance records
- Exporting clean CSV files for PostgreSQL analysis

The cleaned data will later be loaded into PostgreSQL for SQL-based business analysis.

In [185]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

RAW_DIR = Path(r"D:\hr_analytics_raw_dataset\hr_analytics_raw_dataset")
PROJECT_DIR = Path(r"D:\HR-Workforce-Attrition-Analysis")
CLEAN_DIR = PROJECT_DIR / "data" / "clean"

CLEAN_DIR.mkdir(parents=True, exist_ok=True)

AS_OF_DATE = pd.Timestamp("2026-05-01")

RAW_DIR, CLEAN_DIR

(WindowsPath('D:/hr_analytics_raw_dataset/hr_analytics_raw_dataset'),
 WindowsPath('D:/HR-Workforce-Attrition-Analysis/data/clean'))

In [186]:
files = {
    "departments": "raw_departments.csv",
    "job_roles": "raw_job_roles.csv",
    "employees": "raw_employees.csv",
    "salaries": "raw_salaries.csv",
    "performance_reviews": "raw_performance_reviews.csv",
    "satisfaction_surveys": "raw_satisfaction_surveys.csv",
    "training_records": "raw_training_records.csv",
    "attendance": "raw_attendance.csv",
    "attrition_exit_interviews": "raw_attrition_exit_interviews.csv",
    "department_history": "raw_department_history.csv",
}

raw_data = {}

for table_name, file_name in files.items():
    file_path = RAW_DIR / file_name
    raw_data[table_name] = pd.read_csv(file_path, dtype=str, keep_default_na=False)
    print(f"{table_name}: {raw_data[table_name].shape}")

departments: (14, 6)
job_roles: (35, 8)
employees: (1900, 17)
salaries: (5132, 7)
performance_reviews: (4432, 8)
satisfaction_surveys: (3990, 9)
training_records: (7344, 9)
attendance: (54660, 8)
attrition_exit_interviews: (374, 8)
department_history: (2123, 7)


In [187]:
BLANK_VALUES = ["", "null", "NULL", "None", "none", "nan", "NaN", "N/A", "n/a", "NA", "na"]


def clean_text(value):
    """
    Strips whitespace and converts blank-like values to NaN.
    """
    if pd.isna(value):
        return np.nan

    value = str(value).strip()

    if value in BLANK_VALUES:
        return np.nan

    return value


def title_text(value):
    """
    Cleans text and converts it to title case.
    """
    value = clean_text(value)

    if pd.isna(value):
        return np.nan

    return str(value).title()


def parse_numeric(series):
    """
    Converts messy numeric strings to numbers.
    Invalid values become NaN.
    """
    cleaned = (
        series.astype(str)
        .str.replace(",", "", regex=False)
        .str.replace("$", "", regex=False)
        .str.strip()
    )

    cleaned = cleaned.replace(BLANK_VALUES, np.nan)

    return pd.to_numeric(cleaned, errors="coerce")


def parse_date(series):
    """
    Converts messy date strings to datetime.
    Invalid values become NaT.
    """
    cleaned = series.astype(str).str.strip().replace(BLANK_VALUES, np.nan)

    try:
        return pd.to_datetime(cleaned, errors="coerce", format="mixed")
    except TypeError:
        return pd.to_datetime(cleaned, errors="coerce")


def parse_bool(value):
    """
    Standardizes boolean-like values.
    """
    value = clean_text(value)

    if pd.isna(value):
        return np.nan

    value = str(value).strip().lower()

    if value in ["true", "t", "yes", "y", "1"]:
        return True

    if value in ["false", "f", "no", "n", "0"]:
        return False

    return np.nan


def standardize_department(value):
    """
    Standardizes department names.
    """
    value = clean_text(value)

    if pd.isna(value):
        return np.nan

    value_lower = str(value).strip().lower()

    department_map = {
        "hr": "Human Resources",
        "h.r.": "Human Resources",
        "human resources": "Human Resources",
        "people operations": "Human Resources",

        "customer support": "Customer Support",
        "cust support": "Customer Support",
        "support": "Customer Support",
        "client support": "Customer Support",

        "sales": "Sales",
        "sales dept": "Sales",
        "commercial": "Sales",
        "revenue sales": "Sales",

        "engineering": "Engineering",
        "engineerng": "Engineering",
        "software engineering": "Engineering",
        "software": "Engineering",
        "eng": "Engineering",              

        "data": "Data & Analytics",
        "data analytics": "Data & Analytics",
        "data and analytics": "Data & Analytics",
        "analytics": "Data & Analytics",
        "business intelligence": "Data & Analytics",
        "bi": "Data & Analytics",

        "finance": "Finance",
        "fin": "Finance",
        "accounting": "Finance",
        "accounting & finance": "Finance",

        "operations": "Operations",
        "ops": "Operations",
        "operation": "Operations",

        "marketing": "Marketing",
        "mktg": "Marketing",

        "it": "IT Support",
        "it support": "IT Support",
        "information technology": "IT Support",
        "tech support": "IT Support",
    }

    return department_map.get(value_lower, str(value).strip().title())

def standardize_job_title(value):
    """
    Standardizes common job title variations to canonical forms.
    Handles abbreviations, truncations, mixed casing, and typos.
    """
    value = clean_text(value)

    if pd.isna(value):
        return np.nan

    value_lower = str(value).strip().lower()

    job_title_map = {
        # --- Data / BI / Analytics ---
        "data analyst":                     "Data Analyst",
        "bi analyst":                       "Data Analyst",
        "business intelligence analyst":    "Data Analyst",

        "senior data analyst":              "Senior Data Analyst",
        "sr data analyst":                  "Senior Data Analyst",
        "sr. data analyst":                 "Senior Data Analyst",

        "senior bi analyst":                "Senior BI Analyst",
        "sr bi analyst":                    "Senior BI Analyst",
        "bi manager":                       "BI Manager",

        "analytics manager":                "Analytics Manager",
        "analytics mgr":                    "Analytics Manager",

        # --- Software Engineering ---
        "software engineer":                "Software Engineer",
        "software developer":               "Software Engineer",
        "developer":                        "Software Engineer",
        "software engineer i":              "Software Engineer",
        "software engineer ii":             "Software Engineer",
        "software engineer iii":            "Software Engineer",
        "swe":                              "Software Engineer",

        "senior software engineer":         "Senior Software Engineer",
        "sr software engineer":             "Senior Software Engineer",
        "senior developer":                 "Senior Software Engineer",

        "junior software engineer":         "Junior Software Engineer",
        "jr software engineer":             "Junior Software Engineer",
        "jr. software engineer":            "Junior Software Engineer",

        "engineering manager":              "Engineering Manager",
        "eng manager":                      "Engineering Manager",
        "engineering mgr":                  "Engineering Manager",

        # --- Product ---
        "product manager":                  "Product Manager",
        "senior product manager":           "Senior Product Manager",
        "senior pm":                        "Senior Product Manager",
        "sr product manager":               "Senior Product Manager",
        "sr. product manager":              "Senior Product Manager",

        "associate product manager":        "Associate Product Manager",
        "assoc product manager":            "Associate Product Manager",
        "apm":                              "Associate Product Manager",

        # --- Sales ---
        "sales development representative": "Sales Development Representative",
        "sales dev rep":                    "Sales Development Representative",
        "sdr":                              "Sales Development Representative",

        "senior account executive":         "Senior Account Executive",
        "sr account executive":             "Senior Account Executive",
        "account executive":                "Senior Account Executive",
        "senior ae":                        "Senior Account Executive",
        "sr ae":                            "Senior Account Executive",
        "ae":                               "Senior Account Executive",
        "acct executive":                   "Senior Account Executive",

        "sales manager":                    "Sales Manager",
        "sales mgr":                        "Sales Manager",

        # --- Customer Support ---
        "customer support associate":       "Customer Support Associate",
        "support associate":                "Customer Support Associate",
        "customer support agent":           "Customer Support Associate",
        "cust support agent":               "Customer Support Associate",
        "customer service agent":           "Customer Support Associate",

        "customer success manager":         "Customer Success Manager",
        "csm":                              "Customer Success Manager",
        "customer success mgr":             "Customer Success Manager",

        "senior support specialist":        "Senior Support Specialist",
        "sr support specialist":            "Senior Support Specialist",
        "senior support spec.":             "Senior Support Specialist",

        "support team lead":                "Support Team Lead",
        "team lead - support":              "Support Team Lead",
        "support tl":                       "Support Team Lead",

        # --- Operations ---
        "operations analyst":               "Operations Analyst",
        "ops analyst":                      "Operations Analyst",

        "operations coordinator":           "Operations Coordinator",
        "ops coordinator":                  "Operations Coordinator",
        "operations coord.":                "Operations Coordinator",

        "operations manager":               "Operations Manager",
        "ops manager":                      "Operations Manager",

        # --- Finance / Accounting ---
        "finance analyst":                  "Finance Analyst",
        "financial analyst":                "Finance Analyst",
        "fin analyst":                      "Finance Analyst",

        "finance manager":                  "Finance Manager",
        "finance mgr":                      "Finance Manager",

        "accountant ii":                    "Accountant II",
        "senior accountant":                "Senior Accountant",
        "sr accountant":                    "Senior Accountant",

        # --- IT ---
        "it support specialist":            "IT Support Specialist",
        "tech support specialist":          "IT Support Specialist",
        "helpdesk specialist":              "Helpdesk Specialist",

        "it manager":                       "IT Manager",
        "it mgr":                           "IT Manager",

        "systems administrator":            "Systems Administrator",
        "system administrator":             "Systems Administrator",
        "sysadmin":                         "Systems Administrator",

        # --- Marketing ---
        "marketing manager":                "Marketing Manager",
        "marketing mgr":                    "Marketing Manager",
        "mktg coordinator":                 "Marketing Coordinator",
        "marketing coordinator":            "Marketing Coordinator",
        "digital marketing specialist":     "Digital Marketing Specialist",
        "digital marketer":                 "Digital Marketing Specialist",
        "performance marketing specialist": "Performance Marketing Specialist",

        # --- HR / People ---
        "hr business partner":              "HR Business Partner",
        "hrbp":                             "HR Business Partner",
        "people business partner":          "HR Business Partner",

        "hr coordinator":                   "HR Coordinator",
        "human resources coordinator":      "HR Coordinator",
        "hr coord.":                        "HR Coordinator",

        "talent acquisition specialist":    "Talent Acquisition Specialist",
        "ta specialist":                    "Talent Acquisition Specialist",

        "recruiter":                        "Recruiter",

        "hr specialist":                    "HR Specialist",
        "human resources specialist":       "HR Specialist",
    }

    return job_title_map.get(value_lower, str(value).strip().title())



def standardize_employee_status(value):
    """
    Standardizes employee status.
    """
    value = clean_text(value)

    if pd.isna(value):
        return "Unknown"

    value_lower = str(value).strip().lower()

    if value_lower in ["active", "still working", "employed", "current"]:
        return "Active"

    if value_lower in ["left", "terminated", "resigned", "inactive", "exited"]:
        return "Left"

    return str(value).strip().title()


def age_group(age):
    if pd.isna(age):
        return "Unknown"
    if age < 25:
        return "Under 25"
    if age <= 34:
        return "25-34"
    if age <= 44:
        return "35-44"
    if age <= 54:
        return "45-54"
    return "55+"


def performance_category(score):
    if pd.isna(score):
        return "Unknown"
    if score >= 4.5:
        return "High Performer"
    if score >= 3.5:
        return "Strong Performer"
    if score >= 2.5:
        return "Meets Expectations"
    if score >= 1:
        return "Low Performer"
    return "Invalid Score"


def satisfaction_category(score):
    if pd.isna(score):
        return "Unknown"
    if score >= 4:
        return "High Satisfaction"
    if score >= 3:
        return "Medium Satisfaction"
    if score >= 1:
        return "Low Satisfaction"
    return "Invalid Score"


def salary_band(salary_usd):
    """
    Classifies annual-equivalent USD salary into a band.
    Input must be annual_salary_usd (annual-equivalent USD, post currency conversion).
    Thresholds: Low < $30K | Medium < $70K | High < $120K | Executive $120K+
    """
    if pd.isna(salary_usd):
        return "Unknown"
    if salary_usd < 30000:
        return "Low"
    if salary_usd < 70000:
        return "Medium"
    if salary_usd < 120000:
        return "High"
    return "Executive"

In [188]:
cleaning_summary = []


def add_summary(table_name, raw_rows, clean_rows):
    cleaning_summary.append({
        "table_name": table_name,
        "raw_rows": raw_rows,
        "clean_rows": clean_rows,
        "rows_removed": raw_rows - clean_rows,
        "removed_pct": round((raw_rows - clean_rows) / raw_rows * 100, 2) if raw_rows > 0 else 0
    })

In [189]:
departments_raw = raw_data["departments"].copy()
departments = departments_raw.copy()

departments["department_id"] = parse_numeric(departments["department_id"]).astype("Int64")
departments["department_name"] = departments["department_name"].apply(standardize_department)
departments["business_unit"] = departments["business_unit"].apply(title_text)
departments["department_head"] = departments["department_head"].apply(title_text)
departments["region"] = departments["region"].apply(title_text)
departments["active_flag"] = departments["active_flag"].apply(parse_bool)

departments = departments.dropna(subset=["department_id", "department_name"])
departments = departments.drop_duplicates(subset=["department_id"], keep="first")

add_summary("departments", len(departments_raw), len(departments))

departments.head()

,department_id,department_name,business_unit,department_head,region,active_flag
0,1,Customer Support,Client Operations,Maya Haddad,Mena,True
1,2,Sales,Revenue,Omar Khalil,Mena,True
2,3,Engineering,Technology,Ravi Sharma,Global,True
3,4,Data & Analytics,Technology,Lina Nasser,Global,True
4,5,Human Resources,Corporate Services,Sara Mansour,Mena,True


In [190]:
job_roles_raw = raw_data["job_roles"].copy()
job_roles = job_roles_raw.copy()

job_roles["job_role_id"] = parse_numeric(job_roles["job_role_id"]).astype("Int64")
job_roles["job_title"] = job_roles["canonical_title"].where(
    job_roles["canonical_title"].str.strip() != "",
    job_roles["job_title"]
)

job_roles["job_title"] = job_roles["job_title"].apply(standardize_job_title)
job_roles["job_level"] = job_roles["job_level"].apply(title_text)
job_roles["department_id"] = parse_numeric(job_roles["department_id"]).astype("Int64")
job_roles["role_family"] = job_roles["role_family"].apply(title_text)

job_roles["salary_band_min_usd"] = parse_numeric(job_roles["salary_band_min_usd"])
job_roles["salary_band_max_usd"] = parse_numeric(job_roles["salary_band_max_usd"])

job_roles.loc[job_roles["salary_band_min_usd"] < 0, "salary_band_min_usd"] = np.nan
job_roles.loc[job_roles["salary_band_max_usd"] < 0, "salary_band_max_usd"] = np.nan

job_roles = job_roles.drop(columns=["canonical_title"])
job_roles = job_roles.dropna(subset=["job_role_id", "job_title"])
job_roles = job_roles.drop_duplicates(subset=["job_role_id"], keep="first")

add_summary("job_roles", len(job_roles_raw), len(job_roles))

job_roles.head()

,job_role_id,job_title,job_level,department_id,role_family,salary_band_min_usd,salary_band_max_usd
0,101,Customer Support Associate,Junior,1,Support,17000.0,26000.0
1,102,Senior Support Specialist,Senior,1,Support,28000.0,42000.0
2,103,Support Team Lead,Lead,1,Support,43000.0,62000.0
3,104,Customer Success Manager,Manager,1,Customer Success,52000.0,78000.0
4,105,Sales Development Representative,Junior,2,Sales,22000.0,36000.0


In [191]:
employees_raw = raw_data["employees"].copy()
employees = employees_raw.copy()

employees["employee_id"] = parse_numeric(employees["employee_id"]).astype("Int64")
employees["first_name"] = employees["first_name"].apply(title_text)
employees["last_name"] = employees["last_name"].apply(title_text)
employees["full_name"] = employees["first_name"].fillna("") + " " + employees["last_name"].fillna("")
employees["full_name"] = employees["full_name"].str.strip()

employees["gender"] = employees["gender"].str.strip().str.lower().map({
    "male": "Male",
    "m": "Male",
    "man": "Male",
    "female": "Female",
    "f": "Female",
    "woman": "Female",
    "non-binary": "Non-binary",
    "nonbinary": "Non-binary",
    "prefer not say": "Prefer not to say",      
    "prefer not to say": "Prefer not to say"    
}).fillna(employees["gender"].apply(title_text)).fillna("Unknown")

employees["date_of_birth"] = parse_date(employees["date_of_birth"])
employees["hire_date"] = parse_date(employees["hire_date"])

employees.loc[employees["date_of_birth"] > AS_OF_DATE, "date_of_birth"] = pd.NaT
employees.loc[employees["date_of_birth"] < pd.Timestamp("1940-01-01"), "date_of_birth"] = pd.NaT

employees.loc[employees["hire_date"] > AS_OF_DATE, "hire_date"] = pd.NaT
employees.loc[employees["hire_date"] < pd.Timestamp("1980-01-01"), "hire_date"] = pd.NaT

employees["age"] = ((AS_OF_DATE - employees["date_of_birth"]).dt.days / 365.25).round(0)
employees["age"] = employees["age"].astype("Int64")
employees["age_group"] = employees["age"].apply(age_group)

# --- NEW FIX: Nullify impossible ages (under 18 or over 80) caused by typos ---
employees.loc[(employees["age"] < 18) | (employees["age"] > 80), ["date_of_birth", "age", "age_group"]] = [pd.NaT, pd.NA, "Unknown"]

employees["department_id"] = parse_numeric(employees["department_id"]).astype("Int64")
employees["department_name"] = employees["department_name_raw"].apply(standardize_department)

employees["job_role_id"] = parse_numeric(employees["job_role_id"]).astype("Int64")
employees["job_title"] = employees["job_title_raw"].apply(standardize_job_title)

# Fix 'Pm' based on Department
employees.loc[(employees["job_title"].str.lower() == "pm") & (employees["department_name"].isin(["Product", "Product Management"])), "job_title"] = "Product Manager"
employees.loc[(employees["job_title"].str.lower() == "pm") & (~employees["department_name"].isin(["Product", "Product Management"])), "job_title"] = "Project Manager"

# Fix specific Customer Support Titles
employees.loc[(employees["department_name"] == "Customer Support") & (employees["job_title"].str.lower().isin(["cust support agent", "customer support agent"])), "job_title"] = "Customer Support Associate"

# Capitalize "IT" at the beginning of job titles
employees["job_title"] = employees["job_title"].str.replace(r"^It\b", "IT", regex=True)

employees["manager_id"] = parse_numeric(employees["manager_id"]).astype("Int64")

employees["employment_type"] = employees["employment_type"].str.strip().str.lower().map({
    "full time": "Full-time",
    "full-time": "Full-time",
    "ft": "Full-time",
    "part time": "Part-time",
    "part-time": "Part-time",
    "pt": "Part-time",
    "contract": "Contract",
    "contractor": "Contract",
    "intern": "Intern",
    "internship": "Intern"
}).fillna(employees["employment_type"].apply(title_text))

employees["location"] = employees["location"].apply(title_text).fillna("Unknown")
employees["marital_status"] = employees["marital_status"].apply(title_text).fillna("Unknown")
employees["education_level"] = employees["education_level"].apply(title_text).fillna("Unknown")
employees["employee_status"] = employees["status_raw"].apply(standardize_employee_status)

employees["remote_status"] = employees["remote_status"].str.strip().str.lower().map({
    "remote": "Remote",
    "hybrid": "Hybrid",
    "onsite": "On-site",
    "on-site": "On-site",
    "office": "On-site"
}).fillna(employees["remote_status"].apply(title_text)).fillna("Unknown")

# Keep one employee record per employee_id.
# Priority: Active record first, then most recent hire date.
employees["status_priority"] = employees["employee_status"].map({
    "Active": 1,
    "Left": 2,
    "Unknown": 3
}).fillna(4)

employees = employees.sort_values(
    by=["employee_id", "status_priority", "hire_date"],
    ascending=[True, True, False]
)

employees = employees.drop_duplicates(subset=["employee_id"], keep="first")

employees = employees.drop(columns=[
    "department_name_raw",
    "job_title_raw",
    "status_raw",
    "status_priority"
])

employees = employees.dropna(subset=["employee_id"])

add_summary("employees", len(employees_raw), len(employees))

employees.head()

,employee_id,first_name,last_name,gender,date_of_birth,hire_date,department_id,job_role_id,manager_id,employment_type,location,marital_status,education_level,remote_status,full_name,age,age_group,department_name,job_title,employee_status
0,100001,Rania,Darwish,Female,1991-11-09,2015-08-08,3,110,<NA>,Full-time,"Dubai, Uae",Divorced,Master,Unknown,Rania Darwish,34,25-34,Engineering,Software Engineer,Active
1,100002,Carlos,Saleh,Male,1993-04-24,2026-02-02,1,104,<NA>,Temporary,"Amman, Jordan",Single,Diploma,Unknown,Carlos Saleh,33,25-34,Customer Support,Customer Success Manager,Active
2,100003,Anas,Nasser,Male,1993-03-15,2014-01-27,<NA>,131,100002,Contract,"Amman, Jordan",Unknown,Mba,Hybrid,Anas Nasser,33,25-34,IT Support,IT Manager,Active
3,100004,Mariam,Mansour,Female,1982-02-25,2025-12-12,6,120,<NA>,Contract,"Cairo, Egypt",Single,Unknown,Hybrid,Mariam Mansour,44,35-44,Finance,Finance Analyst,Active
4,100005,Ahmad,Hussein,Male,1999-01-28,2021-05-30,1,101,100002,Part-time,"Amman, Jordan",Prefer Not To Say,Bachelor,Remote,Ahmad Hussein,27,25-34,Customer Support,Customer Support Associate,Left


In [192]:
salaries_raw = raw_data["salaries"].copy()
salaries = salaries_raw.copy()

salaries["salary_id"] = parse_numeric(salaries["salary_id"]).astype("Int64")
salaries["employee_id"] = parse_numeric(salaries["employee_id"]).astype("Int64")
salaries["salary_amount"] = parse_numeric(salaries["salary_amount"])
salaries["currency"] = salaries["currency"].str.strip().str.upper()
salaries["effective_date"] = parse_date(salaries["effective_date"])
salaries["bonus_amount"] = parse_numeric(salaries["bonus_amount"])

# ─────────────────────────────────────────────────────────────────────────────
# PAY FREQUENCY NORMALIZATION
# ─────────────────────────────────────────────────────────────────────────────
# Investigation finding: salary_amount stores the annual-equivalent value
# regardless of the pay_frequency label. Confirmed by comparing medians across
# all frequency groups — they are nearly identical (~$33K–$37K). This is
# consistent with a synthetic dataset where pay_frequency is a payroll delivery
# cadence label, not a unit multiplier.
#
# Action: normalize all labels to a canonical set.
# "Biweekly" maps to "Annual" because salary_amount is already annual-equivalent.
#
# NOTE — if this were a live payroll system storing per-period pay amounts:
#   annual_salary_usd = salary_amount_usd * {Biweekly: 26, Monthly: 12, Annual: 1}
# ─────────────────────────────────────────────────────────────────────────────
salaries["pay_frequency"] = salaries["pay_frequency"].str.strip().str.lower().map({
    "monthly":   "Monthly",
    "month":     "Monthly",
    "annual":    "Annual",
    "annually":  "Annual",
    "yearly":    "Annual",
    "biweekly":  "Annual",   # FIX: salary_amount is annual-equivalent (see note above)
    "bi-weekly": "Annual",   # handle hyphenated variant
    "hourly":    "Hourly",
}).fillna(salaries["pay_frequency"].apply(title_text))

# Remove invalid salary and bonus values.
salaries.loc[salaries["salary_amount"] <= 0, "salary_amount"] = np.nan
salaries.loc[salaries["salary_amount"] > 1_000_000, "salary_amount"] = np.nan
salaries.loc[salaries["bonus_amount"] < 0, "bonus_amount"] = np.nan
salaries.loc[salaries["bonus_amount"] > 250_000, "bonus_amount"] = np.nan

# Currency conversion to USD.
# All salary_amount values are treated as annual-equivalent in local currency.
exchange_rates = {
    "USD": 1.00,
    "JOD": 1.41,
    "AED": 0.27,
    "EGP": 0.021,
    "INR": 0.012,
}

salaries["currency"] = salaries["currency"].where(salaries["currency"].isin(exchange_rates.keys()), "USD")
salaries["exchange_rate_to_usd"] = salaries["currency"].map(exchange_rates)

salaries["salary_amount_usd"] = (salaries["salary_amount"] * salaries["exchange_rate_to_usd"]).round(2)
salaries["bonus_amount_usd"]  = (salaries["bonus_amount"]  * salaries["exchange_rate_to_usd"]).round(2)

# Nullify salary records that are implausibly low after currency conversion.
# Values under $5,000 USD are almost certainly conversion errors or data entry mistakes.
salaries.loc[salaries["salary_amount_usd"] < 5_000, "salary_amount_usd"] = np.nan
salaries.loc[salaries["bonus_amount_usd"]  < 0,     "bonus_amount_usd"]  = np.nan

# annual_salary_usd — explicit alias confirming annual-equivalent unit.
# Used as the primary salary field in all SQL analysis queries.
salaries["annual_salary_usd"] = salaries["salary_amount_usd"]

salaries["salary_band"] = salaries["annual_salary_usd"].apply(salary_band)

# ── Verification: confirm normalization is correct ────────────────────────────
# avg_annual_salary_usd should be in a similar range across all pay_frequency
# groups. If Biweekly avg is ~26x higher than Annual avg, normalization is wrong.
print("pay_frequency distribution after normalization:")
print(salaries["pay_frequency"].value_counts())
print("\nAvg annual_salary_usd by pay_frequency (should be similar across all groups):")
print(salaries.groupby("pay_frequency")["annual_salary_usd"].agg(["count", "median", "mean"]).round(0))

# Keep only salary records connected to valid employees.
valid_employee_ids = set(employees["employee_id"].dropna().astype(int))
salaries = salaries[salaries["employee_id"].isin(valid_employee_ids)]

salaries = salaries.dropna(subset=["salary_id", "employee_id"])
salaries = salaries.sort_values(by=["salary_id", "effective_date"], ascending=[True, False])
salaries = salaries.drop_duplicates(subset=["salary_id"], keep="first")

add_summary("salaries", len(salaries_raw), len(salaries))

salaries.head()


pay_frequency distribution after normalization:
pay_frequency
Annual     4289
Monthly     843
Name: count, dtype: int64

Avg annual_salary_usd by pay_frequency (should be similar across all groups):
               count   median     mean
pay_frequency                         
Annual          3764  37067.0  50478.0
Monthly          764  38115.0  49569.0


,salary_id,employee_id,salary_amount,currency,effective_date,bonus_amount,pay_frequency,exchange_rate_to_usd,salary_amount_usd,bonus_amount_usd,annual_salary_usd,salary_band
0,500001,100001,81638.89,USD,2015-08-30,2618.51,Annual,1.0,81638.89,2618.51,81638.89,High
1,500002,100001,84939.47,USD,2015-10-21,14314.65,Annual,1.0,84939.47,14314.65,84939.47,High
2,500003,100001,89251.48,USD,2018-02-17,12654.84,Annual,1.0,89251.48,12654.84,89251.48,High
3,500004,100001,92302.90,USD,2018-09-05,14030.36,Annual,1.0,92302.90,14030.36,92302.90,High
4,500005,100001,98944.26,USD,2026-01-07,6930.63,Annual,1.0,98944.26,6930.63,98944.26,High


In [193]:
performance_raw = raw_data["performance_reviews"].copy()
performance = performance_raw.copy()

performance["review_id"] = parse_numeric(performance["review_id"]).astype("Int64")
performance["employee_id"] = parse_numeric(performance["employee_id"]).astype("Int64")
performance["review_date"] = parse_date(performance["review_date"])

performance["performance_score"] = parse_numeric(performance["performance_score"])
performance["manager_score"] = parse_numeric(performance["manager_score"])
performance["goals_met_percentage"] = parse_numeric(performance["goals_met_percentage"])
performance["promotion_recommended"] = performance["promotion_recommended"].apply(parse_bool)
performance["reviewer_id"] = parse_numeric(performance["reviewer_id"]).astype("Int64")

performance.loc[~performance["performance_score"].between(1, 5), "performance_score"] = np.nan
performance.loc[~performance["manager_score"].between(1, 5), "manager_score"] = np.nan
performance.loc[~performance["goals_met_percentage"].between(0, 100), "goals_met_percentage"] = np.nan

performance["performance_category"] = performance["performance_score"].apply(performance_category)

performance = performance[performance["employee_id"].isin(valid_employee_ids)]

performance = performance.dropna(subset=["review_id", "employee_id"])
performance = performance.sort_values(by=["review_id", "review_date"], ascending=[True, False])
performance = performance.drop_duplicates(subset=["review_id"], keep="first")

add_summary("performance_reviews", len(performance_raw), len(performance))

performance.head()

,review_id,employee_id,review_date,performance_score,manager_score,goals_met_percentage,promotion_recommended,reviewer_id,performance_category
0,700001,100001,2026-02-17,3.22,2.65,50.6,False,100988,Meets Expectations
1,700002,100003,2024-10-07,4.55,4.42,83.3,False,100002,High Performer
2,700003,100004,2026-04-14,3.29,3.58,76.0,False,101545,Meets Expectations
3,700004,100004,2026-04-25,2.63,1.41,43.6,False,100575,Meets Expectations
4,700005,100005,2023-03-29,4.57,4.19,NaN,False,100002,High Performer


In [194]:
satisfaction_raw = raw_data["satisfaction_surveys"].copy()
satisfaction = satisfaction_raw.copy()

satisfaction["survey_id"] = parse_numeric(satisfaction["survey_id"]).astype("Int64")
satisfaction["employee_id"] = parse_numeric(satisfaction["employee_id"]).astype("Int64")
satisfaction["survey_date"] = parse_date(satisfaction["survey_date"])

score_cols = [
    "satisfaction_score",
    "engagement_score",
    "work_life_balance_score",
    "manager_relationship_score",
    "career_growth_score",
    "compensation_satisfaction_score"
]

for col in score_cols:
    satisfaction[col] = parse_numeric(satisfaction[col])
    satisfaction.loc[~satisfaction[col].between(1, 5), col] = np.nan

satisfaction["overall_satisfaction_score"] = satisfaction[score_cols].mean(axis=1).round(2)
satisfaction["satisfaction_category"] = satisfaction["satisfaction_score"].apply(satisfaction_category)

satisfaction = satisfaction[satisfaction["employee_id"].isin(valid_employee_ids)]

satisfaction = satisfaction.dropna(subset=["survey_id", "employee_id"])
satisfaction = satisfaction.sort_values(by=["survey_id", "survey_date"], ascending=[True, False])
satisfaction = satisfaction.drop_duplicates(subset=["survey_id"], keep="first")

add_summary("satisfaction_surveys", len(satisfaction_raw), len(satisfaction))

satisfaction.head()

,survey_id,employee_id,survey_date,satisfaction_score,engagement_score,work_life_balance_score,manager_relationship_score,career_growth_score,compensation_satisfaction_score,overall_satisfaction_score,satisfaction_category
0,900001,100001,2023-10-15,4.0,4.0,3.0,4.0,3.0,4.0,3.67,High Satisfaction
1,900002,100002,2026-04-04,2.0,3.0,3.0,3.0,2.0,2.0,2.50,Low Satisfaction
2,900003,100002,2026-04-10,3.0,3.0,3.0,3.0,3.0,2.0,2.83,Medium Satisfaction
3,900004,100002,2026-04-19,2.0,2.0,3.0,3.0,2.0,2.0,2.33,Low Satisfaction
4,900005,100002,2026-04-23,3.0,NaN,3.0,3.0,3.0,3.0,3.00,Medium Satisfaction


In [195]:
training_raw = raw_data["training_records"].copy()
training = training_raw.copy()

training["training_id"] = parse_numeric(training["training_id"]).astype("Int64")
training["employee_id"] = parse_numeric(training["employee_id"]).astype("Int64")
training["program_name"] = training["program_name"].apply(title_text)
training["training_category"] = training["training_category"].apply(title_text)
training["start_date"] = parse_date(training["start_date"])
training["completion_date"] = parse_date(training["completion_date"])

training["completion_status"] = training["completion_status"].str.strip().str.lower().map({
    "complete": "Completed",
    "completed": "Completed",
    "done": "Completed",
    "passed": "Completed",
    "in progress": "In Progress",
    "ongoing": "In Progress",
    "not started": "Not Started",
    "assigned": "Not Started",
    "failed": "Incomplete",
    "incomplete": "Incomplete",
    "dropped": "Incomplete",
    "drop": "Incomplete"
}).fillna(training["completion_status"].apply(title_text))

training["training_score"] = parse_numeric(training["training_score"])
training["training_hours"] = parse_numeric(training["training_hours"])

training.loc[~training["training_score"].between(0, 100), "training_score"] = np.nan
training.loc[~training["training_hours"].between(0, 200), "training_hours"] = np.nan

training.loc[training["completion_status"] != "Completed", "completion_date"] = pd.NaT
training["completed_flag"] = training["completion_status"].eq("Completed")

training = training[training["employee_id"].isin(valid_employee_ids)]

training = training.dropna(subset=["training_id", "employee_id"])
training = training.sort_values(by=["training_id", "start_date"], ascending=[True, False])
training = training.drop_duplicates(subset=["training_id"], keep="first")

add_summary("training_records", len(training_raw), len(training))

training.head()

,training_id,employee_id,program_name,training_category,start_date,completion_date,completion_status,training_score,training_hours,completed_flag
0,300001,100001,New Hire Onboarding,Onboarding,2023-12-29,2024-02-01,Completed,78.5,7.5,True
1,300002,100001,Advanced Customer Handling,Customer Experience,2025-08-18,2025-08-27,Completed,72.8,12.1,True
2,300003,100001,New Hire Onboarding,Onboarding,2021-03-11,2021-04-05,Completed,74.1,8.2,True
3,300004,100002,Manager Coaching Skills,Leadership,2026-02-27,2026-03-26,Completed,63.1,21.3,True
4,300005,100003,Leadership Essentials,Leadership,2023-06-30,2023-07-02,Completed,79.1,24.4,True


In [196]:
attendance_raw = raw_data["attendance"].copy()
attendance = attendance_raw.copy()

attendance["attendance_id"] = parse_numeric(attendance["attendance_id"]).astype("Int64")
attendance["employee_id"] = parse_numeric(attendance["employee_id"]).astype("Int64")
attendance["attendance_date"] = parse_date(attendance["attendance_date"])

attendance["scheduled_hours"] = parse_numeric(attendance["scheduled_hours"])
attendance["worked_hours"] = parse_numeric(attendance["worked_hours"])

attendance["absence_flag"] = attendance["absence_flag"].apply(parse_bool)

attendance["absence_type"] = attendance["absence_type"].str.strip().str.lower().map({
    "sick": "Sick Leave",
    "sick leave": "Sick Leave",
    "personal": "Personal Leave",
    "personal leave": "Personal Leave",
    "vacation": "Annual Leave",
    "annual leave": "Annual Leave",
    "pto": "Annual Leave",
    "unexcused": "Unexcused",
    "unauthorized": "Unexcused",
    "": "None"
}).fillna(attendance["absence_type"].apply(title_text))

attendance["absence_type"] = attendance["absence_type"].fillna("None")

attendance["late_minutes"] = parse_numeric(attendance["late_minutes"])

attendance.loc[~attendance["scheduled_hours"].between(0, 24), "scheduled_hours"] = np.nan
attendance.loc[~attendance["worked_hours"].between(0, 24), "worked_hours"] = np.nan
attendance.loc[~attendance["late_minutes"].between(0, 600), "late_minutes"] = np.nan

attendance["absence_flag"] = attendance["absence_flag"].fillna(False)
attendance["absence_day_count"] = np.where(attendance["absence_flag"], 1, 0)

attendance["missed_hours"] = np.where(
    attendance["scheduled_hours"] > attendance["worked_hours"],
    attendance["scheduled_hours"] - attendance["worked_hours"],
    0
)

attendance["missed_hours"] = attendance["missed_hours"].round(2)

attendance = attendance[attendance["employee_id"].isin(valid_employee_ids)]

attendance = attendance.dropna(subset=["attendance_id", "employee_id"])
attendance = attendance.sort_values(by=["attendance_id", "attendance_date"], ascending=[True, False])
attendance = attendance.drop_duplicates(subset=["attendance_id"], keep="first")

add_summary("attendance", len(attendance_raw), len(attendance))

attendance.head()

,attendance_id,employee_id,attendance_date,scheduled_hours,worked_hours,absence_flag,absence_type,late_minutes,absence_day_count,missed_hours
0,800001,100001,2025-09-03,7.5,7.7,False,None,0.0,0,0.0
1,800002,100001,2025-09-17,7.5,7.0,False,None,0.0,0,0.5
2,800003,100001,2025-09-25,8.0,0.0,True,Unknown,0.0,1,8.0
3,800004,100001,2025-09-29,8.0,7.7,False,None,0.0,0,0.3
4,800005,100001,2025-10-01,9.0,8.7,False,None,0.0,0,0.3


In [197]:
exit_raw = raw_data["attrition_exit_interviews"].copy()
exit_interviews = exit_raw.copy()

exit_interviews["exit_id"] = parse_numeric(exit_interviews["exit_id"]).astype("Int64")
exit_interviews["employee_id"] = parse_numeric(exit_interviews["employee_id"]).astype("Int64")
exit_interviews["exit_date"] = parse_date(exit_interviews["exit_date"])

# Nullify exit dates before year 2000 — these are data entry errors (e.g. 1900-01-01).
exit_interviews.loc[exit_interviews["exit_date"] < pd.Timestamp("2000-01-01"), "exit_date"] = pd.NaT

# Nullify exit dates that fall before the employee's hire date.
# These are data entry errors — exit cannot logically precede hiring.
emp_hire_dates = employees[["employee_id", "hire_date"]].copy()
exit_interviews = exit_interviews.merge(emp_hire_dates, on="employee_id", how="left")
exit_interviews.loc[
    exit_interviews["exit_date"] < exit_interviews["hire_date"],
    "exit_date"
] = pd.NaT
exit_interviews = exit_interviews.drop(columns=["hire_date"])

exit_interviews["exit_type"] = exit_interviews["exit_type"].str.strip().str.lower().map({
    "voluntary": "Voluntary",
    "resigned": "Voluntary",
    "resignation": "Voluntary",
    "involuntary": "Involuntary",
    "terminated": "Involuntary",
    "termination": "Involuntary"
}).fillna(exit_interviews["exit_type"].apply(title_text))

exit_interviews["exit_reason"] = exit_interviews["exit_reason"].str.strip().str.lower().map({
    "better compensation": "Better Compensation",
    "pay": "Better Compensation",
    "salary": "Better Compensation",
    "career growth": "Career Growth",
    "growth": "Career Growth",
    "promotion": "Career Growth",
    "manager relationship": "Manager Relationship",
    "manager": "Manager Relationship",
    "bad manager": "Manager Relationship",
    "workload": "Workload",
    "burnout": "Workload",
    "stress": "Workload",
    "relocation": "Relocation",
    "moved": "Relocation",
    "personal reasons": "Personal Reasons",
    "personal": "Personal Reasons",
    "poor culture fit": "Poor Culture Fit",
    "culture": "Poor Culture Fit",
    "performance termination": "Performance Termination",
    "low performance": "Performance Termination"
}).fillna(exit_interviews["exit_reason"].apply(title_text))

exit_interviews["rehire_eligible"] = exit_interviews["rehire_eligible"].apply(parse_bool)
exit_interviews["exit_interview_score"] = parse_numeric(exit_interviews["exit_interview_score"])

exit_interviews.loc[
    ~exit_interviews["exit_interview_score"].between(1, 10),
    "exit_interview_score"
] = np.nan

exit_interviews["comments"] = exit_interviews["comments"].apply(clean_text)

exit_interviews = exit_interviews[exit_interviews["employee_id"].isin(valid_employee_ids)]

exit_interviews = exit_interviews.dropna(subset=["exit_id", "employee_id"])
exit_interviews = exit_interviews.sort_values(by=["exit_id", "exit_date"], ascending=[True, False])
exit_interviews = exit_interviews.drop_duplicates(subset=["exit_id"], keep="first")

add_summary("attrition_exit_interviews", len(exit_raw), len(exit_interviews))

exit_interviews.head()

,exit_id,employee_id,exit_date,exit_type,exit_reason,rehire_eligible,exit_interview_score,comments
0,400001,100005,2024-08-07,Voluntary,Poor Culture Fit,True,5.0,Employee felt the culture did not match expect...
1,400002,100007,2025-02-27,Involuntary,Performance Termination,False,1.0,Employment ended due to sustained performance ...
2,400003,100015,2024-03-24,Voluntary,Better Compensation,True,3.0,Employee cited stronger pay package from anoth...
3,400004,100018,2021-09-10,Voluntary,Better Compensation,True,5.0,Employee cited stronger pay package from anoth...
4,400005,100019,2025-06-17,Voluntary,Manager Relationship,True,4.0,Employee mentioned lack of manager support and...


In [198]:
history_raw = raw_data["department_history"].copy()
department_history = history_raw.copy()

department_history["history_id"] = parse_numeric(department_history["history_id"]).astype("Int64")
department_history["employee_id"] = parse_numeric(department_history["employee_id"]).astype("Int64")
department_history["department_id"] = parse_numeric(department_history["department_id"]).astype("Int64")
department_history["job_role_id"] = parse_numeric(department_history["job_role_id"]).astype("Int64")
department_history["start_date"] = parse_date(department_history["start_date"])
department_history["end_date"] = parse_date(department_history["end_date"])
department_history["change_reason"] = department_history["change_reason"].apply(title_text)

# If end date is before start date, treat end date as invalid.
department_history.loc[
    department_history["end_date"] < department_history["start_date"],
    "end_date"
] = pd.NaT

department_history = department_history[department_history["employee_id"].isin(valid_employee_ids)]

department_history = department_history.dropna(subset=["history_id", "employee_id"])
department_history = department_history.sort_values(by=["history_id", "start_date"], ascending=[True, False])
department_history = department_history.drop_duplicates(subset=["history_id"], keep="first")

add_summary("department_history", len(history_raw), len(department_history))

department_history.head()

,history_id,employee_id,department_id,job_role_id,start_date,end_date,change_reason
0,600001,100001,3,110,2015-08-08,NaT,Original Assignment
1,600002,100002,1,104,2026-02-02,NaT,Original Assignment
2,600003,100003,2,106,2014-01-27,2016-01-05,Original Assignment
3,600004,100003,9,131,2016-01-06,NaT,Lateral Transfer
4,600005,100004,6,120,2025-12-12,NaT,Original Assignment


In [199]:
# Backfill missing department_id and department_name in employees
dept_id_map = departments.set_index("department_name")["department_id"].to_dict()
employees["department_id"] = employees["department_id"].fillna(employees["department_name"].map(dept_id_map))

dept_name_map = departments.set_index("department_id")["department_name"].to_dict()
employees["department_name"] = employees["department_id"].map(dept_name_map).fillna(employees["department_name"])

# Backfill missing job_role_id and official job_title in employees
job_roles_lookup = job_roles[["job_role_id", "job_title", "department_id"]].copy()
job_roles_lookup["job_title_lower"] = job_roles_lookup["job_title"].str.lower()

# CRITICAL FIX: Deduplicate the lookup table so the merge doesn't multiply employee rows!
job_roles_lookup = job_roles_lookup.drop_duplicates(subset=["job_title_lower", "department_id"], keep="first")

employees["job_title_lower"] = employees["job_title"].str.lower()
employees = employees.merge(
    job_roles_lookup, 
    on=["job_title_lower", "department_id"], 
    how="left", 
    suffixes=("", "_lookup")
)

employees["job_role_id"] = employees["job_role_id"].fillna(employees["job_role_id_lookup"])
employees["job_title"] = employees["job_title_lookup"].fillna(employees["job_title"])

# Clean up temporary lookup columns
employees = employees.drop(columns=["job_title_lower", "job_role_id_lookup", "job_title_lookup"])

# CRITICAL FIX 2: Final safety net to guarantee employee_id is a strictly unique Primary Key
employees = employees.drop_duplicates(subset=["employee_id"], keep="first")

# Ensure ID formats remain as Int64 after merge
employees["department_id"] = employees["department_id"].astype("Int64")
employees["job_role_id"] = employees["job_role_id"].astype("Int64")

print(f"Backfilled Employee IDs. Final employees shape: {employees.shape}")

Backfilled Employee IDs. Final employees shape: (1850, 20)


In [200]:
clean_tables = {
    "dim_departments": departments,
    "dim_job_roles": job_roles,
    "dim_employees": employees,
    "fact_salaries": salaries,
    "fact_performance_reviews": performance,
    "fact_satisfaction_surveys": satisfaction,
    "fact_training_records": training,
    "fact_attendance": attendance,
    "fact_attrition_exit_interviews": exit_interviews,
    "fact_department_history": department_history,
}

for table_name, df in clean_tables.items():
    output_path = CLEAN_DIR / f"{table_name}.csv"
    df.to_csv(output_path, index=False)
    print(f"Exported {table_name}: {df.shape} -> {output_path}")

Exported dim_departments: (10, 6) -> D:\HR-Workforce-Attrition-Analysis\data\clean\dim_departments.csv
Exported dim_job_roles: (34, 7) -> D:\HR-Workforce-Attrition-Analysis\data\clean\dim_job_roles.csv
Exported dim_employees: (1850, 20) -> D:\HR-Workforce-Attrition-Analysis\data\clean\dim_employees.csv
Exported fact_salaries: (5106, 12) -> D:\HR-Workforce-Attrition-Analysis\data\clean\fact_salaries.csv
Exported fact_performance_reviews: (4432, 9) -> D:\HR-Workforce-Attrition-Analysis\data\clean\fact_performance_reviews.csv
Exported fact_satisfaction_surveys: (3990, 11) -> D:\HR-Workforce-Attrition-Analysis\data\clean\fact_satisfaction_surveys.csv
Exported fact_training_records: (7344, 10) -> D:\HR-Workforce-Attrition-Analysis\data\clean\fact_training_records.csv
Exported fact_attendance: (54660, 10) -> D:\HR-Workforce-Attrition-Analysis\data\clean\fact_attendance.csv
Exported fact_attrition_exit_interviews: (374, 8) -> D:\HR-Workforce-Attrition-Analysis\data\clean\fact_attrition_exit_i

In [201]:
cleaning_summary_df = pd.DataFrame(cleaning_summary)
cleaning_summary_df.to_csv(CLEAN_DIR / "cleaning_summary.csv", index=False)

cleaning_summary_df

,table_name,raw_rows,clean_rows,rows_removed,removed_pct
0,departments,14,10,4,28.57
1,job_roles,35,34,1,2.86
2,employees,1900,1850,50,2.63
3,salaries,5132,5106,26,0.51
4,performance_reviews,4432,4432,0,0.00
5,satisfaction_surveys,3990,3990,0,0.00
6,training_records,7344,7344,0,0.00
7,attendance,54660,54660,0,0.00
8,attrition_exit_interviews,374,374,0,0.00
9,department_history,2123,2123,0,0.00


In [202]:
print("Cleaned table shapes:")
for table_name, df in clean_tables.items():
    print(f"{table_name}: {df.shape}")

Cleaned table shapes:
dim_departments: (10, 6)
dim_job_roles: (34, 7)
dim_employees: (1850, 20)
fact_salaries: (5106, 12)
fact_performance_reviews: (4432, 9)
fact_satisfaction_surveys: (3990, 11)
fact_training_records: (7344, 10)
fact_attendance: (54660, 10)
fact_attrition_exit_interviews: (374, 8)
fact_department_history: (2123, 7)


In [203]:
print("Duplicate primary key checks:")

primary_keys = {
    "dim_departments": "department_id",
    "dim_job_roles": "job_role_id",
    "dim_employees": "employee_id",
    "fact_salaries": "salary_id",
    "fact_performance_reviews": "review_id",
    "fact_satisfaction_surveys": "survey_id",
    "fact_training_records": "training_id",
    "fact_attendance": "attendance_id",
    "fact_attrition_exit_interviews": "exit_id",
    "fact_department_history": "history_id",
}

for table_name, key_col in primary_keys.items():
    df = clean_tables[table_name]
    duplicate_count = df.duplicated(subset=[key_col]).sum()
    print(f"{table_name}.{key_col}: {duplicate_count} duplicates")

Duplicate primary key checks:
dim_departments.department_id: 0 duplicates
dim_job_roles.job_role_id: 0 duplicates
dim_employees.employee_id: 0 duplicates
fact_salaries.salary_id: 0 duplicates
fact_performance_reviews.review_id: 0 duplicates
fact_satisfaction_surveys.survey_id: 0 duplicates
fact_training_records.training_id: 0 duplicates
fact_attendance.attendance_id: 0 duplicates
fact_attrition_exit_interviews.exit_id: 0 duplicates
fact_department_history.history_id: 0 duplicates


In [204]:
print("Foreign key checks against dim_employees:")

fact_tables = {
    "fact_salaries": salaries,
    "fact_performance_reviews": performance,
    "fact_satisfaction_surveys": satisfaction,
    "fact_training_records": training,
    "fact_attendance": attendance,
    "fact_attrition_exit_interviews": exit_interviews,
    "fact_department_history": department_history,
}

valid_employee_ids = set(employees["employee_id"].dropna().astype(int))

for table_name, df in fact_tables.items():
    invalid_fk_count = (~df["employee_id"].isin(valid_employee_ids)).sum()
    print(f"{table_name}: {invalid_fk_count} invalid employee_id values")

Foreign key checks against dim_employees:
fact_salaries: 0 invalid employee_id values
fact_performance_reviews: 0 invalid employee_id values
fact_satisfaction_surveys: 0 invalid employee_id values
fact_training_records: 0 invalid employee_id values
fact_attendance: 0 invalid employee_id values
fact_attrition_exit_interviews: 0 invalid employee_id values
fact_department_history: 0 invalid employee_id values


In [205]:
# =========================================================
# POST-CLEANING VALIDATION
# Run this after all tables are cleaned.
# Expected: 0 non-standard department names, 0 employees aged < 18.
# =========================================================

STANDARD_DEPARTMENTS = {
    "Customer Support", "Engineering", "Sales", "Operations",
    "IT Support", "Finance", "Human Resources", "Data & Analytics",
    "Marketing", "Product"
}

# 1. Check for any non-standard department names in dim_employees
non_standard = employees[~employees["department_name"].isin(STANDARD_DEPARTMENTS)]
print(f"Non-standard department names in dim_employees: {len(non_standard)}")
if len(non_standard) > 0:
    print(non_standard["department_name"].value_counts().to_string())

# 2. Check for any non-standard department names in dim_departments
non_standard_dims = departments[~departments["department_name"].isin(STANDARD_DEPARTMENTS)]
print(f"\nNon-standard department names in dim_departments: {len(non_standard_dims)}")
if len(non_standard_dims) > 0:
    print(non_standard_dims["department_name"].value_counts().to_string())

# 3. Age sanity check
bad_age = employees[(employees["age"].notna()) & ((employees["age"] < 18) | (employees["age"] > 80))]
print(f"\nEmployees with implausible age (< 18 or > 80): {len(bad_age)}")

# 4. hire_date > exit_date cross-check
exit_dates_check = exit_interviews[["employee_id", "exit_date"]].copy()
tenure_check = employees[["employee_id", "hire_date"]].merge(exit_dates_check, on="employee_id", how="inner")
bad_tenure = tenure_check[tenure_check["hire_date"] > tenure_check["exit_date"]]
print(f"\nEmployees where hire_date > exit_date: {len(bad_tenure)}")
if len(bad_tenure) > 0:
    print(bad_tenure[["employee_id", "hire_date", "exit_date"]].to_string())

print("\n✓ Validation complete.")


Non-standard department names in dim_employees: 0

Non-standard department names in dim_departments: 0

Employees with implausible age (< 18 or > 80): 0

Employees where hire_date > exit_date: 0

✓ Validation complete.
